# Day 21: Evaluating Agent Performance

An agent can sound confident and still fail the task. Checking only the final answer is like a driving test that only checks whether you reached the destination, ignoring the route, the mirrors, and the parking. This notebook builds a small evaluation harness that grades agents on task success, tool correctness, forbidden actions, efficiency, and cost, not just on whether the final text looks right.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from eval_tasks import TEST_SET
for task in TEST_SET:
    print(task.task_id, '-', task.difficulty)


## The test set

Eight fixed tasks across support, shopping, and one task that checks whether the agent correctly avoids a forbidden action. Running the exact same tasks every time is what makes scores across agents and versions comparable.

In [ ]:
task = TEST_SET[6]
print('Goal:', task.goal)
print('Expected tools:', task.expected_tools)
print('Forbidden tools:', task.forbidden_tools)


## Scripted agents

Three fake agents stand in for real ones: careful (correct and minimal), sloppy (correct but wasteful), and risky (crosses a line on one task). They call the real Day 16 support and shopping tools.

In [ ]:
from scripted_agents import AGENTS
run = AGENTS['sloppy'](task)
print('Steps:', run['steps'])
for call in run['tool_calls']:
    print(' ', call['tool_name'], call['args'], '->', call['ok'])
print('Answer:', run['final_answer'])


## Graders

Each run is graded on separate criteria: task success, tool correctness, forbidden tool use, efficiency, and tool error rate. Keeping these separate is what reveals a hard fail that a single pass/fail score would hide.

In [ ]:
from graders import grade_run
for name, agent_fn in AGENTS.items():
    r = agent_fn(task)
    g = grade_run(r, task, name)
    print(name, 'success=', g.task_success, 'forbidden=', g.forbidden_call_made, 'efficiency=', round(g.efficiency, 2))


## Run the harness

The harness runs every agent against the full fixed test set and prints a side-by-side summary.

In [ ]:
from eval_harness import run_harness, summarize
for name, agent_fn in AGENTS.items():
    grades = run_harness(agent_fn, name)
    summary = summarize(grades)
    print(name, summary)


## Cost per success

A simple word-count token estimate and an example price show that the sloppy agent costs more per task without completing any more tasks.

In [ ]:
from cost_tracking import estimate_run_cost
for name, agent_fn in AGENTS.items():
    total = 0.0
    for t in TEST_SET:
        r = agent_fn(t)
        _, cost = estimate_run_cost(r, t.goal)
        total += cost
    print(name, 'total cost', round(total, 5), 'cost per task', round(total / len(TEST_SET), 5))


## Regression check

Saves a baseline summary, runs a 'new version' of the agent, and compares pass rates and forbidden-call counts against the baseline, printing PASS or FAIL with the exact metric that dropped.

In [ ]:
import regression_check
regression_check.main()


## Failure analysis

Groups failed runs by reason (missing tool, wrong answer, forbidden tool, too many steps) and shows a full trace for one failing run.

In [ ]:
import failure_analysis
failure_analysis.main()


## Human review sample

Samples a fixed number of runs for a human to review, weighted toward failures and borderline passes, since automatic graders cannot judge tone or spot a new kind of problem nobody thought to grade for yet.

In [ ]:
import human_review_sample
human_review_sample.main()


## Key takeaways

- A confident final answer is not proof of a correct process: check the tool calls too.
- A fixed test set is what makes scores across agents and versions comparable.
- Forbidden actions are hard fails, graded on their own, never averaged into a single score.
- Cost per success matters more than cost per call: wasted steps inflate cost without adding correctness.
- Regression checks compare against a saved baseline, catching drops before release instead of after.
- Automatic graders miss tone and context, so a sample of runs should always go to a human.